#Importing required libraries.

In [ ]:
import math
import inspect
import os
import re
import csv
import json
import time
import random
import urllib.parse
import urllib.request
from collections import Counter
from contextlib import nullcontext
from dataclasses import asdict, dataclass, fields

import torch
import torch.nn as nn
from torch.nn import functional as F

#####Where to save (Colab / Kaggle / local)

In [ ]:
IN_COLAB = os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle/working")
USE_DRIVE = True  # Colab: keep data + checkpoints on Google Drive, so they survive a disconnect

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/TINY_DLM_FROM_SCRATCH'
elif IN_COLAB:
    root_dir = '/content/TINY_DLM_FROM_SCRATCH'
elif IN_KAGGLE:
    root_dir = '/kaggle/working/TINY_DLM_FROM_SCRATCH'
else:
    root_dir = './TINY_DLM_FROM_SCRATCH'

data_dir = os.path.join(root_dir, 'data', 'babi')
checkpts_dir = os.path.join(root_dir, 'check_points')
os.makedirs(data_dir, exist_ok=True)
os.makedirs(checkpts_dir, exist_ok=True)
print("saving everything in", root_dir)

#What is a Decision Model? 🤔

An LLM is a **writer**: it invents the answer word by word, choosing every word from the whole vocabulary.
A decision model is a **judge**: it is given the question **and the possible answers**, and gives every answer a score.

```
context:     mary moved to the bathroom . john went to the hallway . where is mary ?
candidates:  bathroom | hallway

  [CLS] context [SEP] bathroom   ->  Transformer  ->  one vector  ->  score_head  ->  4.1  ┐
  [CLS] context [SEP] hallway    ->  Transformer  ->  one vector  ->  score_head  ->  0.2  ┘ softmax -> 0.98 | 0.02
                                    (same weights)                    (same weights)
```

* The candidates are **plain text**, not special tokens.
* No layer has the number of candidates in its shape: `score_head` is `Linear(n_embd, 1)` and is used once per
  candidate. So the same model can choose among 2, 3 or 10 options.
* Two training stages:
  1. **pre-training**: guess hidden words (`[MASK]`), like BERT -> the model learns the language.
  2. **fine-tuning**: softmax over the candidates' scores, cross-entropy with the right answer -> the model learns to decide.

#The Core of Tiny DLM 🤖

```
for each candidate i:
    [CLS] context [SEP] candidate_i  ->  wte  ->  EBlock x n_layer (RoPE)  ->  ln_f  ->  mean of the candidate's tokens = z_i
all z_i of one decision              ->  EBlock x n_cand_layer (no positions: candidates are a set)  ->  ln_f
                                     ->  score_head  ->  logit_i  ->  softmax over candidates
pre-training only:                       every token vector  ->  mlm_head (tied to wte)  ->  the hidden word
```

#####Configuration

In [ ]:
@dataclass
class ModelConfig:
    """
    Architecture of the model (everything needed to rebuild it from a checkpoint).

    The defaults describe a small Llama-style encoder: RMSNorm, RoPE, SwiGLU, no biases.
    The switches in the second block let you go back to the GPT-2 choices
    (LayerNorm, learned positions, GELU MLP, biases) and compare.
    """

    # ---- size ----
    block_size: int = 128  # longest "[CLS] context [SEP] candidate" sequence, in tokens
    vocab_size: int = 0  # set from the tokenizer before building the model
    n_layer: int = 2  # blocks that read each (context, candidate) pair
    n_cand_layer: int = 1  # blocks that let the candidates of one decision look at each other
    n_head: int = 4  # nh: attention heads
    n_embd: int = 64  # e: embedding size of every token vector
    drop_rate: float = 0.1
    bias: bool = False  # biases in Linear layers and LayerNorms
    pad_id: int = 0  # padding token id, ignored by attention
    sep_id: int = 3  # [SEP] token id, the candidate starts right after it
    pool: str = "candidate"  # one vector per candidate: "candidate" = mean of the candidate's own tokens, "cls" = [CLS]

    # ---- design choices (defaults = Llama-style) ----
    n_kv_head: int = 0  # nkv: 0 -> same as n_head. Fewer K/V heads = grouped-query attention (Llama, Qwen2)
    head_dim: int = 0  # h, size per head: 0 -> n_embd // n_head (e = nh * h)
    norm_type: str = "rmsnorm"  # "rmsnorm" (Llama, Qwen2) or "layernorm" (GPT-2)
    norm_eps: float = 1e-6
    pos_encoding: str = "rope"  # "rope" rotary (Llama, Qwen2) or "learned" absolute positions (GPT-2)
    rope_base: float = 10000.0  # RoPE: theta[m, i] = m / rope_base^(2i / h)  (m position, i pair, h size per head)
    activation: str = "silu"  # "silu" (Llama, inside SwiGLU), "gelu_tanh" (GPT-2), "gelu", "relu"
    gated_mlp: bool = True  # SwiGLU MLP: down(act(gate(x)) * up(x)) (Llama, Qwen2)
    intermediate_size: int = 0  # MLP hidden size, 0 -> 4 * n_embd

    def __post_init__(self):
        assert self.head_dim or self.n_embd % self.n_head == 0, "n_embd % n_head should be 0."
        assert self.n_head % self.kv_heads == 0, "n_head must be a multiple of n_kv_head"
        assert self.norm_type in ("layernorm", "rmsnorm"), self.norm_type
        assert self.pos_encoding in ("learned", "rope"), self.pos_encoding
        assert self.pool in ("candidate", "cls"), self.pool
        assert self.activation in ("gelu_tanh", "gelu", "silu", "relu"), self.activation

    # derived sizes
    @property
    def kv_heads(self):  # nkv
        return self.n_kv_head or self.n_head

    @property
    def head_size(self):  # h
        return self.head_dim or self.n_embd // self.n_head

    @property
    def hidden_size(self):
        return self.intermediate_size or 4 * self.n_embd

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, d):
        """Build a config from a dict, ignoring unknown keys (e.g. old checkpoints)."""
        if "rope_theta" in d:  # old name of rope_base
            d = {**d, "rope_base": d["rope_theta"]}
        names = {f.name for f in fields(cls)}
        return cls(**{k: v for k, v in d.items() if k in names})

#####Layer Norm / RMS Norm (Normalization Layers)

In [ ]:
class LayerNorm(nn.Module):
    """
        Layer Normalization with an optional bias (PyTorch's nn.LayerNorm doesn't support bias=False).
        Used by: GPT-2, BERT.

        y = (X - mean(X)) / sqrt(var(X) + eps) * weight + bias

        Tensor          Type            Shape
        ===========================================================================
        input           float           (b, s, e)      b sequences, s tokens, e embedding size
        ---------------------------------------------------------------------------
        output          float           (b, s, e)      b sequences, s tokens, e embedding size
        ===========================================================================
    """

    def __init__(self, ndim, bias, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.eps = eps

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, self.eps)


class RMSNorm(nn.Module):
    """
        Root Mean Square normalization: no mean subtraction and no bias, a bit cheaper than LayerNorm
        and works as well.   Used by: Llama, Qwen2, ModernBERT.

        y = X / sqrt(mean(X^2) + eps) * weight

        Tensor          Type            Shape
        ===========================================================================
        input           float           (..., n_embd)
        ---------------------------------------------------------------------------
        output          float           (..., n_embd)
        ===========================================================================
    """

    def __init__(self, ndim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.eps = eps

    def forward(self, X):
        dtype = X.dtype
        X = X.float()  # normalise in float32 for stability under mixed precision
        X = X * torch.rsqrt(X.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * X.to(dtype)


def build_norm(config):
    if config.norm_type == "rmsnorm":
        return RMSNorm(config.n_embd, eps=config.norm_eps)
    return LayerNorm(config.n_embd, bias=config.bias, eps=config.norm_eps)

#####RoPE (Rotary Position Embedding)

In [ ]:
def rotate_half(X):
    """[x1, x2] -> [-x2, x1] on the last dimension."""
    x1, x2 = X.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    """
        Rotary Position Embedding (RoPE, https://arxiv.org/abs/2104.09864).
        Instead of adding a position vector to the input, Q and K are rotated by an angle that grows
        with the position, so Q·K only depends on the *distance* between two tokens.
        Used by: Llama, Qwen2, ModernBERT.

            m      token position             0, 1, 2, ...
            i      pair index inside a head   0, 1, ..., h/2 - 1
            theta  rotation angle             theta[m, i] = m / base^(2i / h)

            rotate(X) = X * cos(theta) + rotate_half(X) * sin(theta)

        Tensor          Type            Shape
        ===========================================================================
        Q, K            float           (b, nh, s, h)
        ===========================================================================
    """

    def __init__(self, h, max_positions, base=10000.0):
        super().__init__()
        assert h % 2 == 0, "RoPE needs an even h (dimensions per head)"
        self.h, self.base, self.max_positions = h, base, max_positions
        self.reset_buffers()

    def reset_buffers(self, device=None):
        """(Re)compute the cos/sin tables, they are not saved in checkpoints."""
        i = torch.arange(self.h // 2, device=device).float()  # pair index
        freq = 1.0 / self.base ** (2 * i / self.h)  # (h/2,)
        m = torch.arange(self.max_positions, device=device).float()  # token positions
        theta = torch.outer(m, freq)  # (max_positions, h/2)
        theta = torch.cat((theta, theta), dim=-1)  # (max_positions, h): both halves of a pair share an angle
        self.register_buffer("cos", theta.cos(), persistent=False)
        self.register_buffer("sin", theta.sin(), persistent=False)

    def forward(self, Q, K):
        s = Q.size(-2)
        cos = self.cos[:s].to(Q.dtype)  # (s, h)
        sin = self.sin[:s].to(Q.dtype)
        return Q * cos + rotate_half(Q) * sin, K * cos + rotate_half(K) * sin

#####Bidirectional Self Attention

In [ ]:
class SelfAttention(nn.Module):
    """
        Bidirectional Self-Attention Module
        ( why not causal like an LLM? a decision model reads the whole input at once and generates
          nothing, so every token may look at every other token, before and after it )

        Supports multi-head attention and grouped-query attention (Llama, Qwen2), where several query
        heads share the same key/value head (nkv < nh).

        `keep` marks what may be attended to: real tokens (not padding) inside a pair, or real
        candidates (not empty slots) when the candidates look at each other.

        `use_positions=False` turns RoPE off. Used for the candidate layer: the candidates are an
        unordered set, so their order must not change anything.

            b batch, s sequence length, e embedding size, nh heads, nkv key/value heads, h size per head

        Tensor          Type            Shape
        ===========================================================================
        X               float           (b, s, e)
        keep            bool            (b, s)
        Q               float           (b, nh, s, h)
        K, V            float           (b, nkv, s, h) -> repeated to (b, nh, s, h)
        ---------------------------------------------------------------------------
        output          float           (b, s, e)
        ===========================================================================
    """

    def __init__(self, config, rotary=None, use_positions=True):
        super().__init__()
        self.nh = config.n_head
        self.nkv = config.kv_heads
        self.h = config.head_size
        self.drop_rate = config.drop_rate
        self.split_sizes = [self.nh * self.h, self.nkv * self.h, self.nkv * self.h]

        # Q, K, V projections for all heads, in one matrix
        self.c_attn = nn.Linear(config.n_embd, sum(self.split_sizes), bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(self.nh * self.h, config.n_embd, bias=config.bias)
        self.res_drop = nn.Dropout(config.drop_rate)

        # the model shares one rotary table between all layers; standalone layers build their own
        self.rotary = None
        if use_positions and config.pos_encoding == "rope":
            self.rotary = rotary or RotaryEmbedding(self.h, config.block_size, config.rope_base)

    def forward(self, X, keep):
        b, s, _ = X.size()

        Q, K, V = self.c_attn(X).split(self.split_sizes, dim=2)
        Q = Q.view(b, s, self.nh, self.h).transpose(1, 2)  # (b, nh, s, h)
        K = K.view(b, s, self.nkv, self.h).transpose(1, 2)  # (b, nkv, s, h)
        V = V.view(b, s, self.nkv, self.h).transpose(1, 2)  # (b, nkv, s, h)

        if self.rotary is not None:  # positions go into Q and K only, never into V
            Q, K = self.rotary(Q, K)

        if self.nkv != self.nh:  # grouped-query attention: each K/V head is shared by nh / nkv query heads
            K = K.repeat_interleave(self.nh // self.nkv, dim=1)  # (b, nh, s, h)
            V = V.repeat_interleave(self.nh // self.nkv, dim=1)

        # every query may look at every kept key: (b, 1, 1, s) broadcasts over heads and queries
        mask = keep[:, None, None, :]

        # flash attention when available: softmax(Q K^T / sqrt(h)) V
        context = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask,
                                                 dropout_p=self.drop_rate if self.training else 0.0)  # (b, nh, s, h)
        context = context.transpose(1, 2).contiguous().view(b, s, self.nh * self.h)  # (b, s, e)

        return self.res_drop(self.c_proj(context))

#####MLP (Multi-Layer Perceptron)

In [ ]:
ACTIVATIONS = {
    "gelu_tanh": lambda: nn.GELU(approximate="tanh"),  # GPT-2 ("gelu_new")
    "gelu": nn.GELU,
    "silu": nn.SiLU,  # Llama, Qwen2 (inside SwiGLU)
    "relu": nn.ReLU,
}


class MLP(nn.Module):
    """
        Multi-Layer Perceptron (MLP): computation layer.
        Non-linearity = complex mappings + feature extraction + ... + processing data (reasoning)

            classic (GPT-2):           c_proj( act(c_fc(X)) )
            gated   (Llama, Qwen2):    c_proj( act(c_gate(X)) * c_fc(X) )      a.k.a. SwiGLU with act = SiLU

        Tensor          Type            Shape
        ===========================================================================
        input           float           (b, s, e)      b sequences, s tokens, e embedding size
        ---------------------------------------------------------------------------
        output          float           (b, s, e)      b sequences, s tokens, e embedding size
        ===========================================================================
    """

    def __init__(self, config):
        super().__init__()
        hidden = config.hidden_size
        self.c_fc = nn.Linear(config.n_embd, hidden, bias=config.bias)  # "up" projection
        self.c_gate = nn.Linear(config.n_embd, hidden, bias=config.bias) if config.gated_mlp else None
        self.act = ACTIVATIONS[config.activation]()
        self.c_proj = nn.Linear(hidden, config.n_embd, bias=config.bias)  # "down" projection
        self.dropout = nn.Dropout(config.drop_rate)

    def forward(self, X):
        if self.c_gate is not None:
            hidden = self.act(self.c_gate(X)) * self.c_fc(X)  # (b, s, hidden_size): 4e by default
        else:
            hidden = self.act(self.c_fc(X))  # (b, s, hidden_size): 4e by default
        return self.dropout(self.c_proj(hidden))  # (b, s, e)

#####Block (communication + computation)

In [ ]:
class EBlock(nn.Module):
    """
        Transformer (Encoder) Block: communication layer (att mechanism) + computation layer (feedforward).
        Pre-normalisation, as in GPT-2 and every model after it.

            X = X + attn(ln_1(X), keep)
            X = X + mlp(ln_2(X))

        Same block as tiny-llm's DBlock, except attention is bidirectional (Encoder) instead of
        causal (Decoder), and it takes a `keep` mask for padding.

        Tensor          Type            Shape
        ===========================================================================
        input           float           (b, s, e)      b sequences, s tokens, e embedding size
        keep            bool            (b, s)
        ---------------------------------------------------------------------------
        output          float           (b, s, e)      b sequences, s tokens, e embedding size
        ===========================================================================
    """

    def __init__(self, config, rotary=None, use_positions=True):
        super().__init__()
        self.ln_1 = build_norm(config)
        self.attn = SelfAttention(config, rotary, use_positions)
        self.ln_2 = build_norm(config)
        self.mlp = MLP(config)

    def forward(self, X, keep):
        X = X + self.attn(self.ln_1(X), keep)
        return X + self.mlp(self.ln_2(X))

#####Body of Tiny DLM

In [ ]:
class TinyDLM(nn.Module):
    """
    Rhoa (ῥόα, pomegranate): the text decision model. A Jev-style decision model.

    An LLM *generates* the answer token by token. A decision model gets a list of candidate answers
    and *scores all of them in one forward pass*, then a softmax over the candidates gives one
    probability per candidate. It can only ever pick one of the given candidates.

        for each option j:
            [CLS] context [SEP] option_j  ->  wte  ->  EBlock x n_layer  ->  ln_f  ->  pool = z_j
        all z_j of one decision           ->  EBlock x n_cand_layer (no positions)  ->  ln_f
                                          ->  score_head  ->  logit_j  ->  softmax over the k options

    pool: the mean of the candidate's own tokens (default) or the vector at [CLS] (see `pool`).
    The first part reads each (context, candidate) pair (the NanoJev layout). The second part lets
    the candidates of the same decision look at each other before they are scored.

    Training has two stages (see the two training sections below):
        1. pre-training: the encoder learns the language by guessing hidden words ([MASK]), like BERT.
           Bidirectional encoder -> masked-word prediction, not next-word prediction like an LLM.
        2. fine-tuning: the same encoder + the candidate layers + score_head learn to decide.

    Args:
        config: ModelConfig (vocab_size, block_size, n_layer, n_cand_layer, n_head, n_embd, ...)

    Attributes:
        transformer: ModuleDict that reads each pair (Embeddings, Dropout, Blocks, Norm).
        candidates: ModuleDict that mixes the candidates of one decision (Blocks, Norm).
        score_head: Linear layer, one vector -> one logit (fine-tuning).
        mlm_head: Linear layer, one vector -> a score for every word of the vocabulary (pre-training).
        config: Model configuration.

            b decisions in the batch, k options per decision, s tokens per sequence, e embedding size

        Tensor          Type            Shape
        ===========================================================================
        idx             long            (b, k, s)       one "[CLS] context [SEP] option" per option
        cand_mask       bool            (b, k)          True = real option, False = empty slot
        targets         float           (b, k)          target probabilities
        ---------------------------------------------------------------------------
        logits          float           (b, k)          -inf on empty slots
        ===========================================================================
    """

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size > 0, "set config.vocab_size from the tokenizer"
        assert config.block_size is not None
        self.config = config

        rotary = None
        if config.pos_encoding == "rope":  # one cos/sin table shared by every layer
            rotary = RotaryEmbedding(config.head_size, config.block_size, config.rope_base)
        layers = dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),  # token embeddings
            drop=nn.Dropout(config.drop_rate),
            h=nn.ModuleList([EBlock(config, rotary) for _ in range(config.n_layer)]),  # Block X N
            ln_f=build_norm(config),
        )
        if config.pos_encoding == "learned":
            layers["wpe"] = nn.Embedding(config.block_size, config.n_embd)  # positional embeddings
        self.transformer = nn.ModuleDict(layers)

        # no positions here: shuffling the candidates must only shuffle their scores
        self.candidates = nn.ModuleDict(dict(
            h=nn.ModuleList([EBlock(config, use_positions=False) for _ in range(config.n_cand_layer)]),
            ln_f=build_norm(config),
        ))
        # no bias: adding the same number to every option's score does not change the softmax, it could never learn
        self.score_head = nn.Linear(config.n_embd, 1, bias=False)

        # pre-training head: predict the hidden (masked) words, like BERT
        self.mlm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)
        self.transformer.wte.weight = self.mlm_head.weight  # weight tying (as tiny-llm's lm_head)
        # scaled init of residual projections (GPT-2 paper)
        n_residual_layers = config.n_layer + config.n_cand_layer
        for param_name, param in self.named_parameters():
            if param_name.endswith('c_proj.weight'):
                torch.nn.init.normal_(param, mean=0.0, std=0.02 / math.sqrt(2 * n_residual_layers))

    def get_num_params(self, non_embedding=True):
        """Number of params. Position embeddings are excluded by default."""
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding and "wpe" in self.transformer:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def encode(self, idx):
        """
        The shared Transformer encoder, used by both stages.
        idx:     (b, s) token ids            b sequences, s tokens each
        returns: (b, s, e) one vector of size e per token
        """
        b, s = idx.size()
        assert s <= self.config.block_size, f"Max sequence length is {self.config.block_size}, got {s}"
        keep = idx != self.config.pad_id  # (b, s): padding tokens are never attended to
        X = self.transformer.wte(idx)  # (b, s, e)
        if "wpe" in self.transformer:
            m = torch.arange(s, device=idx.device)  # token positions 0 .. s-1
            X = X + self.transformer.wpe(m)  # (s, e) broadcast over the batch

        X = self.transformer.drop(X)
        for block in self.transformer.h:
            X = block(X, keep)  # (b, s, e)
        return self.transformer.ln_f(X)

    def pool(self, idx, H):
        """
        One vector per (context, option) sequence.
        idx: (b, s) token ids, H: (b, s, e) encoder output -> (b, e)

        "cls":       the vector at [CLS] (position 0), like BERT.
        "candidate": the mean of the vectors of the candidate's own tokens (everything after [SEP]).
                     Why: at the start of training [CLS] looks at all ~70 tokens equally, so the one word
                     that differs between candidates is only ~1/70 of its input and training gets stuck
                     at random guessing. A candidate token starts from its own word and only has to
                     collect the evidence about it from the context.
        """
        if self.config.pool == "cls":
            return H[:, 0]  # (b, e): the vector at position 0
        after_sep = (idx == self.config.sep_id).cumsum(dim=-1) > 0  # (b, s): True from [SEP] on
        is_option = after_sep & (idx != self.config.sep_id) & (idx != self.config.pad_id)  # (b, s)
        w = is_option.unsqueeze(-1).to(H.dtype)  # (b, s, 1): 1 on the option's own tokens
        return (H * w).sum(dim=1) / w.sum(dim=1).clamp(min=1)  # (b, e) mean; empty slots -> zeros

    def forward(self, idx, cand_mask=None, targets=None):
        """
        Two stages, one model:

        pre-training (cand_mask is None): masked-word prediction, like BERT
            idx:     (b, s) token ids where some words were replaced by [MASK]
            targets: (b, s) the original id at masked positions, -1 elsewhere (not part of the loss)
            returns: logits (b, s, vocab_size) and the loss

        fine-tuning / deciding (cand_mask given):
            idx:       (b, k, s) token ids, one "[CLS] context [SEP] option" sequence per option
            cand_mask: (b, k) True for real options, False for empty slots (decisions have different k)
            targets:   (b, k) target probabilities (sum to 1 over the real options), optional
            returns:   logits (b, k) and the loss (None when targets are not given)
        """
        if cand_mask is None:
            logits = self.mlm_head(self.encode(idx))  # (b, s, vocab_size)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss

        b, k, s = idx.size()  # b decisions, k options each, s tokens per sequence

        # 1. read every (context, option) pair as its own sequence -> one vector per option
        pairs = idx.view(b * k, s)  # all options of all decisions in one big batch
        Z = self.pool(pairs, self.encode(pairs)).view(b, k, -1)  # (b, k, e)

        # 2. the options of one decision look at each other (empty slots are hidden)
        for block in self.candidates.h:
            Z = block(Z, cand_mask)  # (b, k, e)
        Z = self.candidates.ln_f(Z)

        # 3. one logit per option; empty slots can never be chosen
        logits = self.score_head(Z).squeeze(-1)  # (b, k)
        logits = logits.masked_fill(~cand_mask, -float('Inf'))

        loss = None
        if targets is not None:
            # cross-entropy with soft targets: -sum_i target_i * log p_i.
            # empty slots have log p = -inf and target 0, and 0 * -inf = NaN, so they are zeroed first.
            log_probs = F.log_softmax(logits, dim=-1).masked_fill(~cand_mask, 0.0)
            loss = -(targets * log_probs).sum(dim=-1).mean()

        return logits, loss

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        # 2D params (matmuls, embeddings) are decayed; biases and norms are not
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)

    @torch.no_grad()
    def decide(self, idx, cand_mask, temperature=0.0):
        """
        Pick one candidate per decision (the decision-model version of `generate`).
        temperature = 0 -> take the most likely candidate; > 0 -> sample (useful to explore).
        returns: choice (b,) the chosen option of each decision, probs (b, k)
        """
        logits, _ = self(idx, cand_mask)
        if temperature == 0:
            probs = F.softmax(logits, dim=-1)
            return probs.argmax(dim=-1), probs
        probs = F.softmax(logits / temperature, dim=-1)
        return torch.multinomial(probs, num_samples=1).squeeze(-1), probs

#Data: bAbI 📚

[bAbI](https://arxiv.org/abs/1502.05698) is a very simple question-answering benchmark (Facebook AI Research).
Every question becomes one decision: the candidates are the places mentioned in the story.

* **decisions (stage 2):** task 1, 10k version: 9,000 train / 1,000 val / 1,000 test
* **pre-training text (stage 1):** the training stories of all 20 tasks, without the answers (6,897 stories, ~594k words)

The download uses the Hugging Face dataset viewer API (plain JSON, no extra library) and is skipped if the files exist.

#####Download and build the dataset

In [ ]:
API = "https://datasets-server.huggingface.co/rows"
MIRROR = "https://huggingface.co/datasets/Muennighoff/babi/resolve/main/babi_train.jsonl"
DATASET = "facebook/babi_qa"
SPLITS = {"train": "train", "validation": "val", "test": "test"}  # bAbI name -> our name
PAGE = 100  # the API returns at most 100 rows per request


def words(text):
    """Lowercase words plus '.' and '?' (sentence ends matter: the *last* move is the one that counts)."""
    return re.findall(r"[a-z0-9']+|[.?]", text.lower())


def fetch_stories(config, split, raw_dir):
    """All stories of one split, cached as JSONL in raw_dir. A story: {"text": [...], "type": [...], "answer": [...]}"""
    path = os.path.join(raw_dir, f"{config}_{split}.jsonl")
    if not os.path.exists(path):
        stories, offset, total = [], 0, None
        while total is None or offset < total:
            query = urllib.parse.urlencode(dict(dataset=DATASET, config=config, split=split, offset=offset, length=PAGE))
            with urllib.request.urlopen(f"{API}?{query}", timeout=60) as r:
                page = json.load(r)
            total = page["num_rows_total"]
            stories += [row["row"]["story"] for row in page["rows"]]
            offset += PAGE
        with open(path, "w", encoding="utf-8") as f:
            f.writelines(json.dumps(s) + "\n" for s in stories)
        print(f"downloaded {config}/{split}: {len(stories)} stories")
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def fetch_mirror(raw_dir):
    """The 1k training split of all 20 tasks: rows of {passage, question, answer, task}, cached in raw_dir."""
    path = os.path.join(raw_dir, "muennighoff_babi_train.jsonl")
    if not os.path.exists(path):
        print(f"downloading {MIRROR}")
        urllib.request.urlretrieve(MIRROR, path)
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def mirror_stories(rows, tasks):
    """
    Rebuild whole stories from the mirror, which stores one row per question with the passage so far.
    A row whose passage does not extend the previous one starts a new story. The sentences added since the
    previous question come first, then the question, so the original order is kept. Answers are dropped.
    """
    stories, prev = [], None
    for r in rows:
        if r["task"] not in tasks:
            prev = None
            continue
        if prev is None or not r["passage"].startswith(prev):
            stories.append([])
            prev = ""
        stories[-1] += [r["passage"][len(prev):], r["question"]]
        prev = r["passage"]
    return [" ".join(words(" ".join(s))) for s in stories]


def questions(story):
    """Yield (story so far, question, answer) for every question line in a story (type 1 = question)."""
    sentences = []
    for text, kind, answer in zip(story["text"], story["type"], story["answer"]):
        if kind == 1:
            yield " ".join(sentences), text, answer.lower()
        else:
            sentences.append(text)


def story_text(story):
    """The whole story as one line: every sentence and question, no answers."""
    return " ".join(words(" ".join(story["text"])))


def to_decision(story_text, question, answer, answer_vocab):
    passage = words(story_text)
    mentioned = list(dict.fromkeys(w for w in passage if w in answer_vocab))  # unique, in order
    candidates = mentioned if (answer in mentioned and len(mentioned) >= 2) else sorted(answer_vocab)
    return {"context": " ".join(passage + words(question)), "candidates": " | ".join(candidates), "answers": answer}


def write(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["context", "candidates", "answers"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"wrote {len(rows)} decisions to {path}")

In [ ]:
def prepare_babi(out_dir, tasks=(1,), pretrain_tasks=range(1, 21), prefix="en-valid-10k"):
    raw_dir = os.path.join(out_dir, "raw")
    os.makedirs(raw_dir, exist_ok=True)

    raw = {split: [] for split in SPLITS}  # split -> [(task, story text, question, answer)]
    for task in tasks:
        for split in SPLITS:
            for story in fetch_stories(f"{prefix}-qa{task}", split, raw_dir):
                raw[split] += [(task, *q) for q in questions(story)]

    # answer vocabulary per task, from the training split only
    answer_vocab = {}
    for task, _, _, answer in raw["train"]:
        answer_vocab.setdefault(task, set()).add(answer)

    for split, name in SPLITS.items():
        rows = [to_decision(s, q, a, answer_vocab[t]) for t, s, q, a in raw[split]]
        write(os.path.join(out_dir, f"{name}.csv"), rows)

    # pre-training text: training stories only (no val / test story is ever seen in stage 1)
    texts = [story_text(s) for t in tasks for s in fetch_stories(f"{prefix}-qa{t}", "train", raw_dir)]
    texts += mirror_stories(fetch_mirror(raw_dir), set(pretrain_tasks) - set(tasks))
    for name, part in (("pretrain_train", [x for i, x in enumerate(texts) if i % 20]),
                       ("pretrain_val", [x for i, x in enumerate(texts) if i % 20 == 0])):
        with open(os.path.join(out_dir, f"{name}.txt"), "w", encoding="utf-8") as f:
            f.writelines(x + "\n" for x in part)
        print(f"wrote {len(part)} stories to {name}.txt")


if not os.path.exists(os.path.join(data_dir, "pretrain_val.txt")):
    prepare_babi(data_dir)
else:
    print("dataset already in", data_dir)

#####Tokenizer (word level, built from the data)

In [ ]:
PAD, UNK, CLS, SEP, MASK = "<pad>", "<unk>", "<cls>", "<sep>", "<mask>"
SPECIALS = [PAD, UNK, CLS, SEP, MASK]  # always ids 0, 1, 2, 3, 4


class WordTokenizer:
    """
        Lowercases, then splits into words and the punctuation "." and "?" (sentence ends carry meaning:
        in bAbI the *last* sentence about a person is the one that counts).
        Words never seen in training become <unk>.

        One model input is a (context, candidate) pair:
            encode_pair("it is raining", "take an umbrella")
            -> [<cls>] it is raining [<sep>] take an umbrella
    """

    def __init__(self, itos):
        assert itos[:len(SPECIALS)] == SPECIALS, "the special tokens must come first"
        self.itos = list(itos)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id, self.unk_id, self.cls_id, self.sep_id, self.mask_id = (self.stoi[t] for t in SPECIALS)

    @classmethod
    def from_texts(cls, texts, min_freq=1, keep=()):
        """
        The vocabulary: every word seen at least `min_freq` times in `texts`, plus every word of `keep`
        (the option texts: a word that can be an answer must never become <unk>, however rare it is).
        """
        counts = Counter(w for text in texts for w in cls.split(text))
        words = {w for w, n in counts.items() if n >= min_freq} | {w for text in keep for w in cls.split(text)}
        return cls(SPECIALS + sorted(words - set(SPECIALS)))  # sorted -> same ids every run

    @staticmethod
    def split(text):
        return re.findall(r"[a-z0-9']+|[.?]", text.lower())

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        return [self.stoi.get(w, self.unk_id) for w in self.split(text)]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if i != self.pad_id)

    def encode_pair(self, context, candidate):
        return [self.cls_id] + self.encode(context) + [self.sep_id] + self.encode(candidate)

    def to_dict(self):
        return {"itos": self.itos}

    @classmethod
    def from_dict(cls, d):
        return cls(d["itos"])

#####Data Loader

In [ ]:
@dataclass
class Decision:
    context: str
    candidates: list
    answers: list


def load_csv(path):
    decisions = []
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            candidates = [c.strip() for c in row["candidates"].split("|")]
            answers = [a.strip() for a in row["answers"].split("|")]
            assert set(answers) <= set(candidates), f"an answer is not among the candidates: {row}"
            decisions.append(Decision(row["context"].strip(), candidates, answers))
    return decisions


def all_texts(decisions):
    """Every context and candidate, to build the tokenizer."""
    for d in decisions:
        yield d.context
        yield from d.candidates


def collate(decisions, tokenizer, device="cpu"):
    """
        Turn a list of decisions into padded tensors.

        Tensor          Type            Shape
        ===========================================================================
        b decisions, k = the most options of any decision in the batch, s = the longest sequence

        idx             long            (b, k, s)
        cand_mask       bool            (b, k)
        targets         float           (b, k)
        ===========================================================================

        Empty candidate slots (a decision with fewer candidates than the largest one in the batch)
        hold a lone [CLS] token, never an all-padding row, which would leave attention nothing to look at.
        Targets are uniform over the correct answers: one right answer -> 1.0, two -> 0.5 each.
    """
    b = len(decisions)
    k = max(len(d.candidates) for d in decisions)
    seqs = [[tokenizer.encode_pair(d.context, c) for c in d.candidates] for d in decisions]
    s = max(len(seq) for row in seqs for seq in row)

    idx = torch.full((b, k, s), tokenizer.pad_id, dtype=torch.long)
    idx[:, :, 0] = tokenizer.cls_id
    cand_mask = torch.zeros(b, k, dtype=torch.bool)
    targets = torch.zeros(b, k)
    for n, d in enumerate(decisions):  # n: decision number, j: option number
        for j, (option, seq) in enumerate(zip(d.candidates, seqs[n])):
            idx[n, j, :len(seq)] = torch.tensor(seq)
            cand_mask[n, j] = True
            targets[n, j] = float(option in d.answers)
        targets[n] /= targets[n].sum()
    return idx.to(device), cand_mask.to(device), targets.to(device)


def sample_candidates(decision, max_candidates, rng=random):
    """
        Keep every right answer plus random wrong options, at most `max_candidates` in total (0 -> keep all).
        Used for training only: CLINC150 has 151 options, and each one is a full pass through the encoder.
        The model never sees a fixed list, so it still works with all 151 at test time (no layer depends on K).

            "translate | transfer | weather | ... (151)"   ->   "weather | translate | pay bill | ... (16)"
    """
    if not max_candidates or len(decision.candidates) <= max_candidates:
        return decision
    wrong = [c for c in decision.candidates if c not in decision.answers]
    n_wrong = max(1, max_candidates - len(decision.answers))
    candidates = list(decision.answers) + rng.sample(wrong, min(n_wrong, len(wrong)))
    rng.shuffle(candidates)  # the right answer must not always be first (the candidate layer ignores order anyway)
    return Decision(decision.context, candidates, decision.answers)


def get_batch(decisions, batch_size, tokenizer, device, rng=random, max_candidates=0):
    """A random batch of decisions: (idx, cand_mask, targets)."""
    batch = rng.sample(decisions, min(batch_size, len(decisions)))
    return collate([sample_candidates(d, max_candidates, rng) for d in batch], tokenizer, device)

> pre-training batches: BERT masking (80% `[MASK]`, 10% random word, 10% unchanged)

In [ ]:
def load_text(path):
    """One text (a story) per line."""
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def make_chunks(texts, tokenizer, block_size):
    """
        Cut every text into pieces of at most block_size tokens, each starting with [CLS]
        (the same start as a decision input, so pre-training and fine-tuning see the same layout).
    """
    n = block_size - 1
    chunks = []
    for text in texts:
        ids = tokenizer.encode(text)
        chunks += [[tokenizer.cls_id] + ids[start:start + n] for start in range(0, len(ids), n)]
    return chunks


def mask_tokens(chunks, tokenizer, mask_prob=0.15, rng=random):
    """
        BERT masking. Pick ~mask_prob of the words (at least one per row); the model must guess them.
        Of the picked words: 80% become [MASK], 10% a random word, 10% stay the same
        (so the model can't just learn "only [MASK] positions matter").

        Tensor          Type            Shape
        ===========================================================================
        b chunks, s = the longest chunk in tokens

        idx             long            (b, s)          the input, with masks
        targets         long            (b, s)          original id where picked, -1 elsewhere
        ===========================================================================
    """
    s = max(len(c) for c in chunks)  # the longest chunk, in tokens
    idx = torch.full((len(chunks), s), tokenizer.pad_id, dtype=torch.long)
    targets = torch.full((len(chunks), s), -1, dtype=torch.long)
    first_word = len(SPECIALS)  # random replacements are real words, never special tokens
    for n, chunk in enumerate(chunks):  # n: row number, m: token position
        idx[n, :len(chunk)] = torch.tensor(chunk)
        positions = list(range(1, len(chunk)))  # never mask [CLS] at m = 0
        picked = [m for m in positions if rng.random() < mask_prob] or [rng.choice(positions)]
        for m in picked:
            targets[n, m] = chunk[m]
            r = rng.random()
            if r < 0.8:
                idx[n, m] = tokenizer.mask_id
            elif r < 0.9:
                idx[n, m] = rng.randrange(first_word, len(tokenizer))
    return idx, targets


def get_mlm_batch(chunks, batch_size, tokenizer, device, mask_prob=0.15, rng=random):
    """
    A random batch for pre-training: (idx, None, targets).
    The None is the cand_mask slot of model.forward(idx, cand_mask, targets): no candidates -> pre-training mode.
    """
    idx, targets = mask_tokens(rng.sample(chunks, min(batch_size, len(chunks))), tokenizer, mask_prob, rng)
    return idx.to(device), None, targets.to(device)

In [ ]:
train_data = load_csv(os.path.join(data_dir, "train.csv"))
val_data = load_csv(os.path.join(data_dir, "val.csv"))
test_data = load_csv(os.path.join(data_dir, "test.csv"))
pretrain_train = load_text(os.path.join(data_dir, "pretrain_train.txt"))
pretrain_val = load_text(os.path.join(data_dir, "pretrain_val.txt"))

# one vocabulary for both stages: every word of the decisions and of the pre-training text
tokenizer = WordTokenizer.from_texts(list(all_texts(train_data)) + pretrain_train)
print(f"{len(train_data)} / {len(val_data)} / {len(test_data)} decisions | {len(pretrain_train)} stories | vocab {len(tokenizer)}")
print(train_data[0])

#Model Configuration and Hyperparameters

In [ ]:
seed = 2000
eval_interval = 250
log_interval = 50
eval_iters = 20

# model
block_size = 128   # longest "[CLS] context [SEP] candidate" (bAbI task 1: 75 tokens)
n_layer = 4        # blocks that read each (context, candidate) pair
n_cand_layer = 1   # blocks where the candidates of one decision look at each other
n_head = 4
n_embd = 128
dropout = 0.1
bias = False
pool = "candidate"  # "candidate" (mean of the candidate's tokens) or "cls" (the [CLS] vector, like BERT)

# stage 1: pre-training (masked words)
pt_batch_size = 64
pt_max_iters = 3000
pt_learning_rate = 1e-3
mask_prob = 0.15

# stage 2: fine-tuning (decisions)
ft_batch_size = 64
ft_max_iters = 3000
ft_learning_rate = 5e-4

# adamw optimizer
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0

# learning rate decay (linear warmup -> cosine -> min_lr)
decay_lr = True
warmup_iters = 100
min_lr_ratio = 0.1  # min_lr = learning_rate * min_lr_ratio

# system
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'

#Initialization

> AUTOMATIC MIXED PRECISION: on the GPU the forward pass runs in float16 / bfloat16 (faster, less memory).
> With float16 a `GradScaler` scales the loss up so small gradients do not round to zero.

In [ ]:
random.seed(seed)
torch.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device == 'cpu' else torch.amp.autocast(device_type=device, dtype=ptdtype)

model_args = dict(vocab_size=len(tokenizer), block_size=block_size, n_layer=n_layer, n_cand_layer=n_cand_layer,
                  n_head=n_head, n_embd=n_embd, drop_rate=dropout, bias=bias, pool=pool,
                  pad_id=tokenizer.pad_id, sep_id=tokenizer.sep_id)
print(f"device: {device} | dtype: {dtype}")

#Utils

> Learning Rate ( Linear warmup -> Cosine Decay -> min_lr )

In [ ]:
# Learning Rate Update
def get_lr(it, learning_rate, max_iters):
    min_lr = learning_rate * min_lr_ratio
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # goes from 1 to 0
    return min_lr + coeff * (learning_rate - min_lr)


# Estimation of Loss
@torch.no_grad()
def estimate_loss(model, get_batch_fns):
    out = {}
    model.eval()
    for split, get_batch_fn in get_batch_fns.items():
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            with ctx:
                _, loss = model(*get_batch_fn())
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


# Accuracy: how often the most likely candidate is a right answer
@torch.no_grad()
def accuracy(model, decisions, batch_size=128):
    model.eval()
    correct = 0
    for i in range(0, len(decisions), batch_size):
        idx, cand_mask, targets = collate(decisions[i:i + batch_size], tokenizer, device)
        with ctx:
            choice, _ = model.decide(idx, cand_mask)
        correct += (targets[torch.arange(len(choice)), choice] > 0).sum().item()
    model.train()
    return correct / len(decisions)


# Save / Load Checkpoints (with the tokenizer, so a checkpoint is all you need to use the model)
def save_checkpts(ckpt_path, model, optimizer, iter_num, best_val_loss):
    checkpoint = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'model_args': model.config.to_dict(),
        'iter_num': iter_num,
        'best_val_loss': best_val_loss,
        'tokenizer': tokenizer.to_dict(),
    }
    torch.save(checkpoint, ckpt_path)
    print(f"Checkpoint saved to {ckpt_path} :)")


def load_checkpts(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
    model = TinyDLM(ModelConfig.from_dict(checkpoint['model_args']))
    model.load_state_dict(checkpoint['model'])
    return model.to(device), checkpoint

#####Training Loop (used by both stages)

In [ ]:
def train_loop(model, get_batch_fns, learning_rate, max_iters, ckpt_path):
    optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device)
    scaler = torch.amp.GradScaler(enabled=(dtype == 'float16' and device == 'cuda'))
    best_val_loss = float('inf')
    iter_num = 0

    X = get_batch_fns['train']()
    t0 = time.time()
    while True:
        lr = get_lr(iter_num, learning_rate, max_iters) if decay_lr else learning_rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # evaluation (Train / val) + checkpoint when val loss improves
        if iter_num % eval_interval == 0 or iter_num == max_iters:
            losses = estimate_loss(model, get_batch_fns)
            print(f"Val-Step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                if iter_num > 0:
                    save_checkpts(ckpt_path, model, optimizer, iter_num, best_val_loss)
        if iter_num >= max_iters:
            break

        # forward -> backward -> optimizing
        with ctx:
            _, loss = model(*X)
        X = get_batch_fns['train']()  # next batch while the GPU works
        scaler.scale(loss).backward()

        # clip the gradient
        if grad_clip != 0.0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        t1 = time.time()
        dt, t0 = t1 - t0, t1
        if iter_num % log_interval == 0:
            print(f"iter {iter_num}: loss {loss.item():.4f}, lr {lr:.2e}, time {dt * 1000:.2f}ms")
        iter_num += 1

    return best_val_loss

#Sanity Checks (before spending GPU time)

Three classic checks, a few seconds each:
1. **loss at init = random guessing**: `mean(log K)` for decisions (K = number of candidates).
2. **any number of candidates**: the same model gives probabilities that sum to 1 for K = 2, 3, 7, 10.
3. **memorise one small batch**: a working network drives the loss to ~0 on 16 fixed decisions.

In [ ]:
torch.manual_seed(seed)
model = TinyDLM(ModelConfig(**{**model_args, 'drop_rate': 0.0})).to(device)

# 1. loss at init
batch = collate(random.sample(train_data, 64), tokenizer, device)
with torch.no_grad():
    _, loss = model(*batch)
print(f"1. loss at init {loss.item():.3f} | random guessing {batch[1].sum(-1).float().log().mean().item():.3f}")

# 2. any number of candidates
places = ["bathroom", "bedroom", "garden", "hallway", "kitchen", "office"]
for K in (2, 3, 7, 10):
    cands = (places * 2)[:K]
    idx, cand_mask, _ = collate([Decision(train_data[0].context, cands, cands[:1])], tokenizer, device)
    with torch.no_grad():
        probs = F.softmax(model(idx, cand_mask)[0], dim=-1)
    print(f"2. K = {K:2d}: probabilities sum to {probs.sum().item():.3f}")

# 3. memorise 16 decisions
small_batch = collate(random.sample(train_data, 16), tokenizer, device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    _, loss = model(*small_batch)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if loss.item() < 0.05:
        break
print(f"3. memorised 16 decisions: loss {loss.item():.3f} after {step + 1} steps")
assert loss.item() < 0.05, "the network cannot memorise 16 examples: something is broken, do not train"
del model, optimizer

#Stage 1: Pre-training (guess the hidden words)

In [ ]:
pt_chunks = {"train": make_chunks(pretrain_train, tokenizer, block_size),
             "val": make_chunks(pretrain_val, tokenizer, block_size)}
pt_batch_fns = {split: (lambda s=split: get_mlm_batch(pt_chunks[s], pt_batch_size, tokenizer, device, mask_prob))
                for split in pt_chunks}

torch.manual_seed(seed)
model = TinyDLM(ModelConfig(**model_args)).to(device)
print(f"number of parameters: {model.get_num_params() / 1e6:.2f}M")

pretrain_ckpt = os.path.join(checkpts_dir, "pretrain.pt")
best = train_loop(model, pt_batch_fns, pt_learning_rate, pt_max_iters, pretrain_ckpt)
print(f"pre-training done, best val loss {best:.4f}")

#Stage 2: Fine-tuning (learn to decide)

Same model, new data and new loss. The encoder starts from the pre-trained weights; the candidate layer and
`score_head` learn now. Loss = `-log p(right candidate)`, the softmax being over the **candidates**, not the vocabulary.

Set `init_from = "scratch"` to fine-tune without pre-training and see what pre-training brings.

In [ ]:
init_from = "pretrain"  # "pretrain" or "scratch"

if init_from == "pretrain":
    model, _ = load_checkpts(pretrain_ckpt)
    print("starting from the pre-trained model", pretrain_ckpt)
else:
    torch.manual_seed(seed)
    model = TinyDLM(ModelConfig(**model_args)).to(device)
    print("starting from scratch (no pre-training)")

ft_batch_fns = {"train": lambda: get_batch(train_data, ft_batch_size, tokenizer, device),
                "val": lambda: get_batch(val_data, ft_batch_size, tokenizer, device)}

finetune_ckpt = os.path.join(checkpts_dir, f"finetune_{init_from}.pt")
best = train_loop(model, ft_batch_fns, ft_learning_rate, ft_max_iters, finetune_ckpt)

model, _ = load_checkpts(finetune_ckpt)  # the best checkpoint (lowest val loss)
print(f"fine-tuning done, best val loss {best:.4f}")
print(f"accuracy: train {accuracy(model, train_data):.1%} | val {accuracy(model, val_data):.1%} | test {accuracy(model, test_data):.1%}")

#Ask the Model 🗳️

The same model, the same story, different numbers of candidates (it was trained with 2 to 6):

In [ ]:
def ask(context, candidates):
    idx, cand_mask, _ = collate([Decision(context, candidates, candidates[:1])], tokenizer, device)
    choice, probs = model.decide(idx, cand_mask)
    print(f"context: {context}")
    for cand, p in sorted(zip(candidates, probs[0].tolist()), key=lambda x: -x[1]):
        print(f"   {p:5.2f}  {cand}")
    print(f"decision: {candidates[choice.item()]}\n")


story = "mary moved to the bathroom . john went to the hallway . mary went to the garden . where is mary ?"
ask(story, ["bathroom", "garden"])
ask(story, ["bathroom", "hallway", "garden"])
ask(story, ["bathroom", "hallway", "garden", "kitchen", "office"])